In [1]:
import numpy as np

from Util.Problems import Problem, solution


class P014(Problem):
    number = 14
    title = "Longest Collatz Sequence"
    description = """<p>The following iterative sequence is defined for the set of positive integers:</p><ul style="list-style-type:none;">
<li>$n \\to n/2$ ($n$ is even)</li>
<li>$n \\to 3n + 1$ ($n$ is odd)</li></ul><p>Using the rule above and starting with $13$, we generate the following sequence:
$$13 \\to 40 \\to 20 \\to 10 \\to 5 \\to 16 \\to 8 \\to 4 \\to 2 \\to 1.$$</p><p>It can be seen that this sequence (starting at $13$ and finishing at $1$) contains $10$ terms. Although it has not been proved yet (Collatz Problem), it is thought that all starting numbers finish at $1$.</p><p>Which starting number, under one million, produces the longest chain?</p><p class="note"><b>NOTE:</b> Once the chain starts the terms are allowed to go above one million.</p>"""
    upper_bound = 1000000

In [2]:
p = P014()
p.describe()

## Problem 14: Longest Collatz Sequence

<p>The following iterative sequence is defined for the set of positive integers:</p><ul style="list-style-type:none;">
<li>$n \to n/2$ ($n$ is even)</li>
<li>$n \to 3n + 1$ ($n$ is odd)</li></ul><p>Using the rule above and starting with $13$, we generate the following sequence:
$$13 \to 40 \to 20 \to 10 \to 5 \to 16 \to 8 \to 4 \to 2 \to 1.$$</p><p>It can be seen that this sequence (starting at $13$ and finishing at $1$) contains $10$ terms. Although it has not been proved yet (Collatz Problem), it is thought that all starting numbers finish at $1$.</p><p>Which starting number, under one million, produces the longest chain?</p><p class="note"><b>NOTE:</b> Once the chain starts the terms are allowed to go above one million.</p>

### Solution notes
We start with a very basic implementation which recursively calls a collatz_length function and tracks the length and result for all numbers up to 1 million.

In [3]:
@solution(P014, first=True, max_tests= 0, make_fast=False, warmup_args=(P014.upper_bound,))
def brute_force_recursion(upper_bound):
    longest_chain_length = 0
    longest_chain_number = 0
    def collatz_length(number, current_length = 0):
        current_length += 1
        if number == 1:
            return current_length
        if number % 2 == 0:
            return collatz_length(number // 2, current_length)
        else:
            return collatz_length(3 * number + 1, current_length)
    for i in range(1, upper_bound):
        length = collatz_length(i)
        if length > longest_chain_length:
            longest_chain_length = length
            longest_chain_number = i
    return longest_chain_number

In [4]:
p.test_once("brute_force_recursion")

837799 found after a separate test in 7802.481500 ms by brute_force_recursion (first)


The same simple recursion but in numba

In [5]:
@solution(P014, max_tests= 10, make_fast=True, warmup_args=(P014.upper_bound,))
def numba_brute_force_recursion(upper_bound):
    longest_chain_length = 0
    longest_chain_number = 0
    for i in range(1, upper_bound):
        length = 0
        number = i
        while True:
            length += 1
            if number == 1:
                break
            if number % 2 == 0:
                number = number // 2
            else:
                number = 3 * number + 1
        if length > longest_chain_length:
            longest_chain_length = length
            longest_chain_number = i
    return longest_chain_number

In [6]:
p.test_all()

837799 found after a separate test in 7802.481500 ms by brute_force_recursion (first)
837799 found after 10 tests in 140.865650 ms by numba_brute_force_recursion


Now we save all values and their length in a dictionary, and use it to look up already calculated values. This doesn't speed the algorithm up yet because dictionaries are very slow.

In [7]:
@solution(P014, max_tests = 10, make_fast=True, warmup_args=(P014.upper_bound,))
def dict_recursion(upper_bound):
    longest_chain_length = 0
    longest_chain_number = 0
    calculated_lengths = {}
    for i in range(1, upper_bound):
        length = 0
        number = i
        while True:
            # print(calculated_lengths.keys())
            if number in calculated_lengths:
                length += calculated_lengths[number]
                break
            length += 1
            if number == 1:
                break
            if number % 2 == 0:
                number = number // 2
            else:
                number = 3 * number + 1
        calculated_lengths[i] = length
        if length > longest_chain_length:
            longest_chain_length = length
            longest_chain_number = i
    return longest_chain_number

In [8]:
p.test_all()

837799 found after a separate test in 7802.481500 ms by brute_force_recursion (first)
837799 found after 10 tests in 193.718710 ms by dict_recursion
837799 found after 10 tests in 131.286710 ms by numba_brute_force_recursion


As the dictionary was so slow, we do the same thing, but with an array. The size of this array is $1.02\times$ the upper bound. This has been found by trial and error to yield the best results, but anywhere between 1,001 and 1,01 seems similar. In this array we can store the calculated lengths at corresponding indices. This still gives the increased peed of lookup over calculation, but without the expensive dict.

In [9]:
@solution(P014, make_fast=True, warmup_args=(P014.upper_bound,))
def array_recursion(upper_bound):
    longest_chain_length = 0
    longest_chain_number = 0
    array_size = upper_bound + (upper_bound // 50)
    calculated_lengths = np.zeros(array_size, dtype=np.int_)
    for i in range(1, upper_bound):
        length = 0
        number = i
        while True:
            if number < array_size - 1:
                if calculated_lengths[number]:
                    length += calculated_lengths[number]
                    break
            length += 1
            if number == 1:
                break
            if number % 2 == 0:
                number = number // 2
            else:
                number = 3 * number + 1
        calculated_lengths[i] = length
        if length > longest_chain_length:
            longest_chain_length = length
            longest_chain_number = i
    return longest_chain_number

In [10]:
p.test_all(repeats= 100)

837799 found after 100 tests in 12.931162 ms by array_recursion
837799 found after a separate test in 7802.481500 ms by brute_force_recursion (first)
837799 found after 10 tests in 159.071190 ms by dict_recursion
837799 found after 10 tests in 133.303310 ms by numba_brute_force_recursion


Same as last time, but replaced the calculations with binary. Checking if a number is even is done by looking at the last bit, division by 2 is done by bit shifting to the right. $\times 3 + 1$ is done by a bitshift to the left, with a 1 inserted on the right to achieve $\times 2 + 1$ and added to the number once more.

In [47]:
@solution(P014, make_fast=True, warmup_args=(P014.upper_bound,))
def binary_array_recursion(upper_bound):
    longest_chain_length = 0
    longest_chain_number = 0
    array_size = upper_bound + (upper_bound // 50)
    calculated_lengths = np.zeros(array_size, dtype=np.int_)
    for i in range(1, upper_bound):
        length = 0
        number = i
        while True:
            if number < array_size - 1:
                if calculated_lengths[number]:
                    length += calculated_lengths[number]
                    break
            length += 1
            if number == 1:
                break
            if not number & 1: # Bitwise check for even numbers (if the last bit is 1, a number is odd)
                number = number >> 1 # Bitwise division by 2 (shift all of the bits to the right 1 place)
            else:
                number = number + ((number << 1) | 1) # Bitwise *3 + 1, as the number shifted 1 digit to the left with a
                                                        # 1 on the right is the same as the number * 2 + 1
        calculated_lengths[i] = length
        if length > longest_chain_length:
            longest_chain_length = length
            longest_chain_number = i
    return longest_chain_number

In [49]:
p.test_all(repeats= 1000)

837799 found after 1000 tests in 10.877157 ms by array_recursion
837799 found after 1000 tests in 10.487846 ms by binary_array_recursion
837799 found after a separate test in 7802.481500 ms by brute_force_recursion (first)
837799 found after 10 tests in 136.297810 ms by dict_recursion
837799 found after 10 tests in 124.869590 ms by numba_brute_force_recursion
